# Testes da Arquitetura QKD (inspirada ETSI GS QKD 015)

Este notebook valida as funcionalidades implementadas:

- BB84 como produtor de chave para o sistema.
- Registro de sessoes QKD com metadados.
- Buffer de chaves por enlace.
- Operacoes de descoberta, monitoramento e controle no Controller.
- Politica de reposicao por limiar (estoque minimo).

In [1]:
import sys
from pprint import pprint

sys.path.append('../')

from quantumnet.topology import Network
from quantumnet.control import Controller

In [2]:
# Setup de rede basico para os testes.
net = Network()
net.set_ready_topology('Linha', 4)
ctrl = Controller(net)
ctrl.register_routing_tables()

print('Topologia pronta. Nos:', sorted(net.hosts.keys()))
print('Enlaces:', [tuple(sorted(e)) for e in net.edges])

Topologia pronta. Nos: [0, 1, 2, 3]
Enlaces: [(0, 1), (1, 2), (2, 3)]


In [3]:
# 1) Descoberta de recursos QKD.
qkd_nodes = ctrl.get_qkd_nodes()
qkd_links = ctrl.get_qkd_links()

print('QKD Nodes:')
pprint(qkd_nodes)
print('\nQKD Links (estado inicial):')
pprint(qkd_links)

assert len(qkd_nodes) == 4, 'Numero de nos inesperado.'
assert len(qkd_links) == 3, 'Numero de enlaces inesperado em Linha(4).'

QKD Nodes:
[0, 1, 2, 3]

QKD Links (estado inicial):
[{'bits_available': 0,
  'key_rate_bps': 0.0,
  'link': (0, 1),
  'state': 'active',
  'supported_protocols': ['BB84']},
 {'bits_available': 0,
  'key_rate_bps': 0.0,
  'link': (1, 2),
  'state': 'active',
  'supported_protocols': ['BB84']},
 {'bits_available': 0,
  'key_rate_bps': 0.0,
  'link': (2, 3),
  'state': 'active',
  'supported_protocols': ['BB84']}]


In [4]:
# 2) Estado inicial do buffer em um enlace direto.
alice, bob = 0, 1
state_before = ctrl.get_buffer_state(alice, bob)
metrics_before = ctrl.get_link_metrics(alice, bob)

print('Buffer antes da sessao BB84:')
pprint(state_before)
print('\nMetricas antes da sessao BB84:')
pprint(metrics_before)

assert state_before['bits_available'] == 0, 'Buffer deveria iniciar vazio.'

Buffer antes da sessao BB84:
{'bits_available': 0,
 'link': (0, 1),
 'min_bits_threshold': 128,
 'state': 'active'}

Metricas antes da sessao BB84:
{'bits_available': 0,
 'key_rate_bps': 0.0,
 'link': (0, 1),
 'min_bits_threshold': 128,
 'state': 'active',
 'successful_sessions': 0,
 'supported_protocols': ['BB84'],
 'total_generated_bits': 0,
 'total_sessions': 0}


In [5]:
# 3) Controle: iniciar sessao BB84 e validar producao de chave para buffer.
# Usa retentativas para evitar falha esporadica por natureza estocastica da simulacao.
requested_bits = 32
max_attempts = 5
bb84_result = None

for attempt in range(1, max_attempts + 1):
    bb84_result = ctrl.start_bb84_session(alice, bob, requested_bits)
    if bb84_result is not None:
        print(f'BB84 bem-sucedido na tentativa {attempt}.')
        break
    print(f'Tentativa {attempt} falhou, tentando novamente...')

print('Resultado BB84:')
pprint(bb84_result)

assert bb84_result is not None, 'BB84 falhou apos retentativas neste teste.'
assert 'key' in bb84_result and len(bb84_result['key']) == requested_bits, 'Tamanho de chave inesperado.'

BB84 bem-sucedido na tentativa 1.
Resultado BB84:
{'avg_qber': 0.16772486772486772,
 'buffered_bits': 32,
 'key': [1,
         0,
         1,
         1,
         1,
         0,
         0,
         0,
         1,
         1,
         0,
         0,
         1,
         0,
         0,
         1,
         0,
         1,
         0,
         1,
         1,
         1,
         0,
         0,
         1,
         1,
         0,
         0,
         0,
         1,
         0,
         0],
 'qber_history': [0.5,
                  0.16666666666666666,
                  0.2,
                  0.14285714285714285,
                  0.5,
                  0.0,
                  0.0,
                  0.0,
                  0.0],
 'rounds': 10,
 'session_id': 1}


In [6]:
# 4) Monitoramento apos BB84: buffer e metricas devem refletir a sessao.
state_after = ctrl.get_buffer_state(alice, bob)
metrics_after = ctrl.get_link_metrics(alice, bob)
sessions = ctrl.get_qkd_sessions()

print('Buffer apos BB84:')
pprint(state_after)
print('\nMetricas apos BB84:')
pprint(metrics_after)
print('\nUltima sessao registrada:')
pprint(sessions[-1])

assert state_after['bits_available'] >= requested_bits, 'Buffer nao foi abastecido corretamente.'
assert metrics_after['total_sessions'] >= 1, 'Total de sessoes deveria aumentar.'
assert sessions[-1]['protocol'] == 'BB84', 'Protocolo da sessao deve ser BB84.'
assert sessions[-1]['participants'] == (alice, bob), 'Participantes da sessao incorretos.'
assert sessions[-1]['generated_bits'] == requested_bits, 'Bits gerados na sessao nao conferem.'

Buffer apos BB84:
{'bits_available': 32,
 'link': (0, 1),
 'min_bits_threshold': 128,
 'state': 'active'}

Metricas apos BB84:
{'bits_available': 32,
 'key_rate_bps': 3.2,
 'link': (0, 1),
 'min_bits_threshold': 128,
 'state': 'active',
 'successful_sessions': 1,
 'supported_protocols': ['BB84'],
 'total_generated_bits': 32,
 'total_sessions': 1}

Ultima sessao registrada:
{'avg_qber': 0.16772486772486772,
 'ended_at': 10,
 'generated_bits': 32,
 'participants': (0, 1),
 'protocol': 'BB84',
 'requested_bits': 32,
 'rounds': 10,
 'session_id': 1,
 'started_at': 0,
 'status': 'completed'}


In [7]:
# 5) Requisicao de chave do buffer (consumo).
consume_bits = 12
key_chunk = ctrl.request_key(alice, bob, consume_bits)
state_after_consume = ctrl.get_buffer_state(alice, bob)

print('Quantidade consumida:', len(key_chunk))
print('Buffer apos consumo:')
pprint(state_after_consume)

assert len(key_chunk) == consume_bits, 'Quantidade retornada pelo buffer incorreta.'
assert state_after_consume['bits_available'] == state_after['bits_available'] - consume_bits, 'Consumo do buffer inconsistente.'

Quantidade consumida: 12
Buffer apos consumo:
{'bits_available': 20,
 'link': (0, 1),
 'min_bits_threshold': 128,
 'state': 'active'}


In [8]:
# 6) Gerenciamento: politica por limiar (estoque minimo).
# Forca threshold alto para disparar reposicao no enlace (0,1).
ctrl.set_minimum_stock(alice, bob, 80)
actions = ctrl.ensure_minimum_stock(default_replenish_bits=64)
state_after_policy = ctrl.get_buffer_state(alice, bob)
metrics_after_policy = ctrl.get_link_metrics(alice, bob)

print('Acoes da politica de reposicao:')
pprint(actions)
print('\nBuffer apos politica:')
pprint(state_after_policy)
print('\nMetricas apos politica:')
pprint(metrics_after_policy)

assert any(a['link'] == (alice, bob) for a in actions), 'Politica nao avaliou o enlace esperado.'
assert metrics_after_policy['total_sessions'] >= metrics_after['total_sessions'], 'Total de sessoes nao deveria diminuir.'
assert state_after_policy['bits_available'] >= state_after_consume['bits_available'], 'Reposicao nao aumentou/estabilizou estoque.'

Acoes da politica de reposicao:
[{'available_before': 20,
  'link': (0, 1),
  'requested_replenish_bits': 64,
  'status': 'started',
  'threshold': 80},
 {'available_before': 0,
  'link': (1, 2),
  'requested_replenish_bits': 128,
  'status': 'started',
  'threshold': 128},
 {'available_before': 0,
  'link': (2, 3),
  'requested_replenish_bits': 128,
  'status': 'started',
  'threshold': 128}]

Buffer apos politica:
{'bits_available': 84,
 'link': (0, 1),
 'min_bits_threshold': 80,
 'state': 'active'}

Metricas apos politica:
{'bits_available': 84,
 'key_rate_bps': 8.0,
 'link': (0, 1),
 'min_bits_threshold': 80,
 'state': 'active',
 'successful_sessions': 2,
 'supported_protocols': ['BB84'],
 'total_generated_bits': 96,
 'total_sessions': 2}


In [9]:
# 7) Relatorio final resumido.
final_sessions = ctrl.get_qkd_sessions()
final_state = ctrl.get_buffer_state(alice, bob)
final_metrics = ctrl.get_link_metrics(alice, bob)

summary = {
    'sessions_recorded': len(final_sessions),
    'last_session_id': final_sessions[-1]['session_id'] if final_sessions else None,
    'buffer_bits_available': final_state['bits_available'],
    'link_total_sessions': final_metrics['total_sessions'],
    'link_successful_sessions': final_metrics['successful_sessions'],
    'supported_protocols': final_metrics['supported_protocols'],
}

print('Resumo final de validacao:')
pprint(summary)

assert 'BB84' in summary['supported_protocols'], 'Enlace deve suportar BB84.'
print('\nTodos os checks passaram com sucesso.')

Resumo final de validacao:
{'buffer_bits_available': 84,
 'last_session_id': 4,
 'link_successful_sessions': 2,
 'link_total_sessions': 2,
 'sessions_recorded': 4,
 'supported_protocols': ['BB84']}

Todos os checks passaram com sucesso.


In [10]:
# Resumo compacto para avaliacao rapida
print({
    'sessions_recorded': len(final_sessions),
    'link_total_sessions': final_metrics['total_sessions'],
    'link_successful_sessions': final_metrics['successful_sessions'],
    'buffer_bits_available': final_state['bits_available'],
    'supported_protocols': final_metrics['supported_protocols'],
})

{'sessions_recorded': 4, 'link_total_sessions': 2, 'link_successful_sessions': 2, 'buffer_bits_available': 84, 'supported_protocols': ['BB84']}
